# Analisis Time Series, Ekstraksi Fitur, PCA, dan K-Means Clustering

**Proyek Sains Data (PSD)**  
**Polutan:** CO, NO₂, SO₂  

## Catatan ruang lingkup

Instruksi kerja yang sedang digunakan menyatakan bahwa ekstraksi dilakukan untuk **3 polutan sekaligus**:

- CO → 68 fitur
- NO₂ → 68 fitur
- SO₂ → 68 fitur

Total dataset utama untuk clustering:

**68 × 3 = 204 fitur per mahasiswa.**

In [2]:
# 1. Import library

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:.6g}")


# 4. EDA — Time Series CO, NO₂, dan SO₂

Setiap polutan dibuat sebagai time series. Data yang digunakan untuk visualisasi berasal dari hasil preprocessing sehingga tidak memiliki missing value.


In [ ]:
# 4.1 Membaca time series

ts_no2 = pd.read_csv(NO2_TS, parse_dates=["date"])
ts_co = pd.read_csv(CO_TS, parse_dates=["date"])
ts_so2 = pd.read_csv(SO2_TS, parse_dates=["date"])

print("NO2:", ts_no2.shape, "missing =", ts_no2["NO2"].isna().sum())
print("CO :", ts_co.shape, "missing =", ts_co["CO"].isna().sum())
print("SO2:", ts_so2.shape, "missing =", ts_so2["SO2"].isna().sum())


In [ ]:
# 4.2 Plot time series NO2

plt.figure(figsize=(14, 5))
plt.plot(ts_no2["date"], ts_no2["NO2"])
plt.title("Time Series NO₂ — Kecamatan Menganti")
plt.xlabel("Tanggal")
plt.ylabel("NO₂")
plt.grid(True)
plt.show()


In [ ]:
# 4.3 Plot time series CO

plt.figure(figsize=(14, 5))
plt.plot(ts_co["date"], ts_co["CO"])
plt.title("Time Series CO — Kecamatan Menganti")
plt.xlabel("Tanggal")
plt.ylabel("CO")
plt.grid(True)
plt.show()


In [ ]:
# 4.4 Plot time series SO2

plt.figure(figsize=(14, 5))
plt.plot(ts_so2["date"], ts_so2["SO2"])
plt.title("Time Series SO₂ — Kecamatan Menganti")
plt.xlabel("Tanggal")
plt.ylabel("SO₂")
plt.grid(True)
plt.show()


# 5. Konsep statistik dasar

Statistik deskriptif yang digunakan untuk membaca karakteristik data antara lain:

- **Mean**: rata-rata aritmetika
- **Median**: nilai tengah setelah data diurutkan
- **Minimum / maksimum**: nilai ekstrem bawah dan atas
- **Range**: selisih maksimum dan minimum
- **Variance**: ukuran penyebaran kuadrat terhadap mean
- **Standard deviation**: akar dari variance
- **Quartile / IQR**: ukuran sebaran bagian tengah data
- **Skewness**: kemiringan distribusi
- **Kurtosis**: keruncingan distribusi

Perhitungan manual di bawah menggunakan contoh kecil agar langkah matematis mudah diperiksa.


In [ ]:
# 5.1 Contoh hitungan manual statistik

x = np.array([2, 4, 5, 4, 9])

n = len(x)
mean_manual = np.sum(x) / n

x_sorted = np.sort(x)
median_manual = x_sorted[n // 2]

variance_population_manual = np.sum((x - mean_manual) ** 2) / n
std_population_manual = np.sqrt(variance_population_manual)

range_manual = np.max(x) - np.min(x)

print("Data :", x.tolist())
print("Mean :", mean_manual)
print("Median :", median_manual)
print("Range :", range_manual)
print("Variance populasi :", variance_population_manual)
print("Std populasi :", std_population_manual)


## Contoh perhitungan manual — Mean

Untuk data:

\(x = [2,4,5,4,9]\)

\[
\bar{x} = \frac{2+4+5+4+9}{5}
= \frac{24}{5}
= 4.8
\]

Jadi mean = **4.8**.


# 6. Fitur TSFEL

TSFEL menghasilkan fitur yang dikelompokkan ke domain statistik, temporal, dan spectral. Dokumen contoh TSFEL yang digunakan dalam proyek ini membahas 68 fitur dan memberikan rumus serta contoh manual untuk masing-masing fitur.

Untuk tugas individual Verdi, fitur yang dibagikan adalah:

## Hurst Exponent

Hurst Exponent digunakan untuk menggambarkan karakter dependensi/persistensi suatu deret waktu. Penjelasan rumus dan metode manual harus disesuaikan dengan metode yang digunakan oleh TSFEL pada versi environment proyek.

**Verifikasi:** nilai manual dan nilai TSFEL dibandingkan pada data yang sama.

> Bagian ini akan diisi setelah kita cek implementasi `hurst_exponent` yang tepat pada versi TSFEL di environment proyek, sehingga rumus manual tidak berbeda dengan fungsi yang benar-benar dipanggil.


In [ ]:
# 6.1 Verifikasi Hurst Exponent dengan TSFEL

import inspect
import tsfel.feature_extraction.features as tsfel_features

print(inspect.signature(tsfel_features.hurst_exponent))
print(inspect.getsource(tsfel_features.hurst_exponent))


# 7. Membaca data hasil ekstraksi seluruh kelas

Setiap file mempunyai identitas:

- `id`
- `nama`
- `daerah`

serta 68 fitur TSFEL.

Dataset utama 204 fitur akan dibuat dengan menggabungkan ketiga polutan berdasarkan identitas mahasiswa.


In [ ]:
# 7.1 Load tiga file kelas

df_no2 = pd.read_csv(NO2_CLASS)
df_co = pd.read_csv(CO_CLASS)
df_so2 = pd.read_csv(SO2_CLASS)

print("NO2:", df_no2.shape)
print("CO :", df_co.shape)
print("SO2:", df_so2.shape)

print("\nKolom identitas:")
print(df_no2[["id", "nama", "daerah"]].head())


In [ ]:
# 7.2 Validasi kesamaan identitas mahasiswa

keys = ["id", "nama", "daerah"]

ids_no2 = df_no2[keys].astype(str)
ids_co = df_co[keys].astype(str)
ids_so2 = df_so2[keys].astype(str)

print("Identitas NO2 == CO :", ids_no2.equals(ids_co))
print("Identitas NO2 == SO2:", ids_no2.equals(ids_so2))


In [ ]:
# 7.3 Menggabungkan menjadi 204 fitur

feature_cols_no2 = [c for c in df_no2.columns if c not in keys]
feature_cols_co = [c for c in df_co.columns if c not in keys]
feature_cols_so2 = [c for c in df_so2.columns if c not in keys]

no2_renamed = df_no2[keys + feature_cols_no2].copy()
co_renamed = df_co[keys + feature_cols_co].copy()
so2_renamed = df_so2[keys + feature_cols_so2].copy()

no2_renamed = no2_renamed.rename(columns={c: f"NO2_{c}" for c in feature_cols_no2})
co_renamed = co_renamed.rename(columns={c: f"CO_{c}" for c in feature_cols_co})
so2_renamed = so2_renamed.rename(columns={c: f"SO2_{c}" for c in feature_cols_so2})

df_204 = (
    no2_renamed
    .merge(co_renamed, on=keys, how="inner", validate="one_to_one")
    .merge(so2_renamed, on=keys, how="inner", validate="one_to_one")
)

feature_cols_204 = [c for c in df_204.columns if c not in keys]

print("Shape dataset 204 fitur:", df_204.shape)
print("Jumlah fitur:", len(feature_cols_204))
print("Missing:", df_204[feature_cols_204].isna().sum().sum())


# 8. Standardisasi fitur

K-Means menggunakan jarak terhadap centroid. Karena 204 fitur memiliki skala yang berbeda, fitur distandardisasi menggunakan Z-score melalui `StandardScaler`.

\[
z = \frac{x-\mu}{\sigma}
\]

Setelah standardisasi, setiap fitur berada pada skala yang sebanding untuk perhitungan jarak.


In [ ]:
scaler_204 = StandardScaler()
X_204 = df_204[feature_cols_204].astype(float)

X_204_scaled = scaler_204.fit_transform(X_204)

print("Shape:", X_204_scaled.shape)
print("Mean mendekati 0:", np.mean(X_204_scaled, axis=0).mean())
print("Std rata-rata mendekati 1:", np.std(X_204_scaled, axis=0).mean())


# 9. PCA 1–37

Sesuai instruksi tugas, kualitas clustering dibandingkan pada jumlah komponen PCA dari 1 sampai 37.

Untuk setiap jumlah komponen:
1. transformasi PCA
2. cari kandidat K dengan Elbow
3. evaluasi K-Means
4. simpan metrik clustering

**Catatan:** hasil K terbaik akan ditentukan dari data yang benar-benar digunakan, bukan menyalin K dari contoh teman.


In [ ]:
# 9.1 PCA 1 sampai 37 — tabel explained variance

max_components = min(37, X_204_scaled.shape[0], X_204_scaled.shape[1])

pca_full = PCA(n_components=max_components)
X_pca_full = pca_full.fit_transform(X_204_scaled)

explained = pd.DataFrame({
    "PC": np.arange(1, max_components + 1),
    "explained_variance_ratio": pca_full.explained_variance_ratio_,
    "cumulative_explained_variance": np.cumsum(pca_full.explained_variance_ratio_)
})

display(explained)


In [ ]:
# 9.2 Fungsi evaluasi K-Means untuk suatu jumlah PCA

def evaluate_kmeans(X, k_min=2, k_max=10, random_state=42):
    rows = []

    upper_k = min(k_max, len(X) - 1)

    for k in range(k_min, upper_k + 1):
        model = KMeans(
            n_clusters=k,
            random_state=random_state,
            n_init=10
        )

        labels = model.fit_predict(X)

        rows.append({
            "K": k,
            "inertia": model.inertia_,
            "silhouette": silhouette_score(X, labels),
            "davies_bouldin": davies_bouldin_score(X, labels),
            "calinski_harabasz": calinski_harabasz_score(X, labels)
        })

    return pd.DataFrame(rows)


In [ ]:
# 9.3 Evaluasi PCA 1–37

all_pca_results = []

for n_pc in range(1, max_components + 1):
    X_pc = X_pca_full[:, :n_pc]
    result = evaluate_kmeans(X_pc)
    result.insert(0, "n_pca", n_pc)
    all_pca_results.append(result)

pca_kmeans_results = pd.concat(all_pca_results, ignore_index=True)

display(pca_kmeans_results.head())


## Cara membaca metrik

- **Inertia**: semakin kecil, jarak dalam cluster semakin kecil; digunakan terutama untuk Elbow Method.
- **Silhouette Score**: semakin mendekati 1 berarti pemisahan cluster semakin jelas; nilai negatif menunjukkan banyak titik lebih dekat ke cluster lain.
- **Davies-Bouldin Index**: semakin kecil menunjukkan cluster yang lebih kompak dan lebih terpisah.
- **Calinski-Harabasz Index**: semakin besar menunjukkan rasio pemisahan antar-cluster terhadap penyebaran dalam-cluster yang lebih tinggi.

Tidak boleh menentukan “K terbaik” hanya dari satu angka tanpa melihat konteks metrik dan struktur cluster.


In [ ]:
# 9.4 Pilih kandidat K berdasarkan silhouette tertinggi untuk setiap PCA

best_by_pca = (
    pca_kmeans_results
    .sort_values(["n_pca", "silhouette"], ascending=[True, False])
    .groupby("n_pca", as_index=False)
    .first()
)

display(best_by_pca)


In [ ]:
# 9.5 Visualisasi perubahan silhouette terhadap jumlah PCA

plt.figure(figsize=(12, 5))
plt.plot(
    best_by_pca["n_pca"],
    best_by_pca["silhouette"],
    marker="o"
)
plt.xlabel("Jumlah Komponen PCA")
plt.ylabel("Silhouette Score")
plt.title("Kualitas Clustering terhadap Jumlah Komponen PCA")
plt.grid(True)
plt.show()


# 10. Menentukan K menggunakan Elbow Method

Untuk K yang dipilih pada PCA tertentu, plot inertia untuk K=2 sampai K=10.

Pola “siku” menjadi kandidat K, kemudian diperiksa lagi menggunakan metrik evaluasi clustering.


In [ ]:
# Contoh: gunakan jumlah PCA yang dipilih setelah melihat hasil bagian 9.
# Jangan mengisi manual sebelum tabel best_by_pca diperiksa.

candidate_pca = int(best_by_pca.loc[best_by_pca["silhouette"].idxmax(), "n_pca"])
X_candidate = X_pca_full[:, :candidate_pca]

elbow_candidate = evaluate_kmeans(X_candidate)

display(elbow_candidate)

plt.figure(figsize=(8, 5))
plt.plot(elbow_candidate["K"], elbow_candidate["inertia"], marker="o")
plt.xlabel("Jumlah Cluster (K)")
plt.ylabel("Inertia")
plt.title(f"Elbow Method — PCA {candidate_pca}")
plt.grid(True)
plt.show()


# 11. K-Means final pada dataset 204 fitur

Setelah jumlah komponen PCA dan K ditentukan dari hasil evaluasi, model final dijalankan dan label cluster ditambahkan ke identitas mahasiswa.

Nilai `candidate_pca` dan `final_k` harus berasal dari hasil analisis pada bagian sebelumnya.


In [ ]:
# Isi final_k setelah pemeriksaan Elbow + metrik.
# Contoh berikut mengambil K dengan silhouette tertinggi pada PCA kandidat.

final_k = int(
    elbow_candidate.loc[elbow_candidate["silhouette"].idxmax(), "K"]
)

final_model = KMeans(
    n_clusters=final_k,
    random_state=42,
    n_init=10
)

labels_204 = final_model.fit_predict(X_candidate)

hasil_cluster_204 = df_204[keys].copy()
hasil_cluster_204["cluster"] = labels_204

print("PCA digunakan:", candidate_pca)
print("K digunakan  :", final_k)

display(hasil_cluster_204)


In [ ]:
# 11.1 Evaluasi final

final_silhouette = silhouette_score(X_candidate, labels_204)
final_db = davies_bouldin_score(X_candidate, labels_204)
final_ch = calinski_harabasz_score(X_candidate, labels_204)

print("Silhouette Score      :", final_silhouette)
print("Davies-Bouldin Index  :", final_db)
print("Calinski-Harabasz     :", final_ch)


# 12. Analisis 68 fitur

Tahap ini diulang menggunakan 68 fitur dalam satu polutan, mengikuti pola pekerjaan contoh teman. Analisis dapat dilakukan terpisah untuk NO₂, CO, dan SO₂.

Untuk setiap polutan:
1. pilih 68 fitur
2. StandardScaler
3. PCA
4. Elbow Method
5. K-Means
6. Silhouette / DBI / CH
7. visualisasi dan profiling cluster


In [ ]:
def run_68_feature_analysis(df, pollutant, identity_cols=("id", "nama", "daerah")):
    identity_cols = list(identity_cols)
    feature_cols = [c for c in df.columns if c not in identity_cols]

    X = df[feature_cols].astype(float)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    n_components = min(37, X_scaled.shape[0], X_scaled.shape[1])
    pca = PCA(n_components=n_components)
    X_pca = pca.fit_transform(X_scaled)

    return {
        "identity": df[identity_cols].copy(),
        "feature_cols": feature_cols,
        "X_scaled": X_scaled,
        "X_pca": X_pca,
        "pca_model": pca
    }

analysis_no2 = run_68_feature_analysis(df_no2, "NO2")
analysis_co = run_68_feature_analysis(df_co, "CO")
analysis_so2 = run_68_feature_analysis(df_so2, "SO2")

print("Analisis 68 fitur siap untuk NO2, CO, dan SO2.")


# 13. Kesimpulan

Bagian akhir notebook harus merangkum:

1. karakteristik time series CO, NO₂, dan SO₂
2. proses preprocessing dan ekstraksi fitur
3. pembentukan dataset 204 fitur
4. hasil PCA 1–37
5. kandidat jumlah cluster K
6. hasil evaluasi clustering
7. hasil analisis 68 fitur
8. interpretasi cluster berdasarkan identitas daerah
9. keterbatasan analisis

**Catatan:** kesimpulan numerik diisi setelah seluruh cell analisis dijalankan dan hasil metrik sudah tersedia.
